# RITA at Matched Relative Push — the last unrepaired model

## The gap this closes

After `34`'s residual-norm audit exposed the α_rel confound, three models were re-tested at
genuinely matched relative push: **ZymCTRL** (`38`, then `40`'s threshold search) and **p-IgGen**
(`41`). **RITA was never re-run.** Its §1k numbers are still the original absolute-norm ones and are
uninterpretable for the same reason ZymCTRL's were.

How badly it was over-pushed, from `34`'s own measurements:

| | ‖h‖ | v_L norm forced to | α_rel at nominal "1x" | vs ProtGPT2's 0.20 anchor |
|---|---|---|---|---|
| RITA layer 3 | 23.69 | 583.998 (196×) | **24.652** | **123× too hard** |
| RITA layer 11 | 43.03 | 583.998 (109×) | **13.572** | **68× too hard** |

At α_rel ≈ 25, the injected vector is **25 times larger than the hidden state it is added to** —
that is not steering, it is overwriting the forward pass. §1k's RITA result (CONTROL 90% → 96% →
100%) tells us nothing about steering at a comparable dose.

## Design, and why it differs from `38`/`41`

**Layer 3 only, with a wide dose ladder — not two layers at 1x/2x.** Two deliberate departures:

1. **Skip layer 11.** The early-vs-late layer question is **retracted** (§1i-LAYER-REPAIRED, tested
   on its strongest case and killed). There is no live claim that needs a second layer, so the
   budget is better spent on dose resolution. Layer 3 (25% depth) is also the closer match to
   ProtGPT2's layer 12 of 36 (33%) than layer 11 (100%) is.
2. **Doses 0 / 1× / 2× / 4× / 8× at matched α_rel.** ZymCTRL showed nothing at 1× or 2× and only
   became significant at **3×** (§1h-THRESHOLD). Testing RITA at only 1×/2× would very likely
   reproduce that null and require a second session. Going wide immediately finds the threshold if
   one exists.

## The measurement problem specific to RITA, and the fix

**RITA's natural collapse rate is ~90%.** That leaves only ~10 percentage points of headroom before
the binary collapse metric saturates — it is a nearly useless readout on this model, and §1k's
"90% → 96% → 100%" is mostly ceiling.

**So this notebook reports mean pLDDT as the PRIMARY outcome and collapse rate as secondary.**
pLDDT is continuous and does not saturate: steering can push mean pLDDT from 52 → 35 while the
collapse rate barely moves from 90% → 96%. Statistical test is Mann-Whitney U on the pLDDT
distributions (non-parametric, no normality assumption), with Fisher's exact on collapse retained
for comparability with every other section.

**Entropy is also reported at every dose** — answering a question raised but never tested: *does
steering at a matched dose actually achieve its intended repetition-reduction effect, or does it do
nothing until it does damage?* On ZymCTRL and p-IgGen entropy was flat at matched doses, meaning
those runs showed neither benefit nor harm. RITA's wider ladder can show whether entropy ever moves
before pLDDT falls.

## RITA-specific handling (all five known issues, verbatim from `32`/`33`/`44`)

Scoped monkeypatch with save/restore, forced `float32`, hook path `model.transformer.layers[L]`,
hand-written sampling loop (RITA has no `.generate()`), and forward-hook activation capture
(`output_hidden_states` does not return the standard tuple). Fold-success is tracked per condition
per the §1l lesson.

Kaggle setup: Accelerator = **GPU T4 x1**, Internet = **ON**. Expect ~2–2.5 hours (manual generation
is slower per token than `.generate()`; 250 folds).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import gc, math, collections, urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, EsmForProteinFolding
from scipy.stats import mannwhitneyu, fisher_exact

torch.manual_seed(2026); np.random.seed(2026)
device = "cuda" if torch.cuda.is_available() else "cpu"

REFERENCE_NORM = 583.998      # ProtGPT2's own natural v_L norm; the historical absolute reference
RITA_LAYER = 3                # 25% depth — closest match to ProtGPT2's layer 12 of 36 (33%)
DOSES = [0.0, 1.0, 2.0, 4.0, 8.0]
N_PER_CONDITION = 50
N_CANDIDATES = 200

def clear_gpu():
    gc.collect(); torch.cuda.empty_cache()

def calculate_entropy(s):
    if not s: return 0.0
    c = collections.Counter(s); n = len(s)
    return -sum((v / n) * math.log2(v / n) for v in c.values())

print("CUDA:", torch.cuda.is_available())
print(f"Doses at matched alpha_rel: {DOSES} | layer {RITA_LAYER} | "
      f"{len(DOSES) * N_PER_CONDITION} condition folds + {N_CANDIDATES} pool folds")


CUDA: True
Doses at matched alpha_rel: [0.0, 1.0, 2.0, 4.0, 8.0] | layer 3 | 250 condition folds + 200 pool folds


In [2]:
# --- Probe set + anchor measurement, identical method to 38/40/41 so alpha_rel is on the same
#     scale as every other repair in this project. ---

UNIPROT = ["P0CG48","P00720","P02144","P42212","P01308","P61823",
           "P00648","P99999","P69905","P68871","P00698","P00441"]
FALLBACK = ["NLYIQWLKDGGPSSGRPPPS","LSDEDFKAVFGMTRSAFANLPLWKQQHLKKEKGLF","GSQIGAKNTGQVQLNLLAL",
            "MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYDDATKTFTVTE",
            "MKTIIALSYIFCLVFADYKDDDDKLEHTHHHEASGGNLQVQLQESGGGLVQAGGSLRLSCAASGRTFSNYAMGWFRQAPGKEREFVAAISWSGGSTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAASRFRYWGQGTQVTVSS",
            "DEPPQSPWDRVKDFATVYVDAVKPTGKGKV"]

print("Fetching UniProt reference sequences...")
refs = []
for acc in UNIPROT:
    try:
        with urllib.request.urlopen(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=10) as r:
            lines = [l for l in r.read().decode("utf-8").strip().split("\n") if l]
        s = "".join(lines[1:])
        if len(s) >= 20: refs.append((acc, s))
    except Exception as e:
        print(f"  skip {acc}: {e}")
if not refs:
    print("!! UniProt fetch failed (Internet toggle OFF?). Using hardcoded fallback.")
    refs = [(f"local{i+1}", s) for i, s in enumerate(FALLBACK)]
print(f"{len(refs)} source proteins.")

def build_probe_set(n=40, frag_len=50, seed=7):
    rng = np.random.RandomState(seed)
    out = []
    for i in range(n):
        _, s = refs[i % len(refs)]
        L = min(frag_len, len(s))
        st = rng.randint(0, max(1, len(s) - L + 1))
        out.append(s[st:st + L])
    return out

def build_prefix_pool(n, seed=11):
    rng = np.random.RandomState(seed)
    out = []
    for i in range(n):
        _, s = refs[i % len(refs)]
        pl = rng.randint(10, 16)
        st = rng.randint(0, max(1, len(s) - pl))
        out.append(s[st:st + pl])
    return out

probe_seqs = build_probe_set()

def measure_resid_norm_hooked(model, tokenizer, seqs, layer_module):
    cap = {}
    def hook(m, i, o):
        cap["h"] = (o[0] if isinstance(o, (tuple, list)) else o).detach()
    handle = layer_module.register_forward_hook(hook)
    vals = []
    try:
        for s in seqs:
            ids = tokenizer(s, return_tensors="pt", truncation=True, max_length=256).input_ids.to(device)
            cap.clear()
            with torch.no_grad():
                model(ids)
            if "h" in cap:
                vals.append(cap["h"].float().norm(dim=-1).mean().item())
    finally:
        handle.remove()
    return float(np.mean(vals)) if vals else float("nan")

# ProtGPT2 anchor
print(f"\nLoading ProtGPT2 to measure the anchor...")
a_tok = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
a_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device).eval()
h_protgpt2 = measure_resid_norm_hooked(a_model, a_tok, probe_seqs, a_model.transformer.h[12])
anchor_alpha_rel = REFERENCE_NORM / h_protgpt2
print(f"ProtGPT2 layer-12 ‖h‖ = {h_protgpt2:.2f}  ->  anchor_alpha_rel = {anchor_alpha_rel:.4f}")
print(f"(38/40/41 all measured ~0.2000 — a close value confirms cross-run stability)")
del a_model, a_tok; clear_gpu()


Fetching UniProt reference sequences...
12 source proteins.

Loading ProtGPT2 to measure the anchor...


config.json:   0%|          | 0.00/850 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: nferruz/ProtGPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ProtGPT2 layer-12 ‖h‖ = 2919.92  ->  anchor_alpha_rel = 0.2000
(38/40/41 all measured ~0.2000 — a close value confirms cross-run stability)


In [3]:
# --- Scoring + length-aware ESMFold, unchanged from 39/40/41/44. ---
ALPHABET_SIZE = 20
VALID_AA = set("ACDEFGHIKLMNPQRSTVWY")

def h_norm(s):
    if not s: return 0.0
    c = collections.Counter(s); n = len(s)
    return (-sum((v / n) * math.log2(v / n) for v in c.values())) / math.log2(ALPHABET_SIZE)

def distinct_n(s, n):
    if len(s) < n: return 1.0
    g = [s[i:i+n] for i in range(len(s)-n+1)]
    return len(set(g)) / len(g)

def homopolymer_runs(s):
    if not s: return []
    runs, r = [], 1
    for i in range(1, len(s)):
        if s[i] == s[i-1]: r += 1
        else: runs.append(r); r = 1
    runs.append(r); return runs

def r_hpoly(s, k=4):
    if not s: return 1.0
    return max(0.0, 1.0 - sum(l for l in homopolymer_runs(s) if l >= k) / len(s))

def repetition_score(s):
    return float(np.mean([h_norm(s), distinct_n(s,2), distinct_n(s,3), r_hpoly(s)]))

def utility_score(p, t):
    return float(np.mean([p/100.0, t]))

class SafeEvaluator:
    def __init__(self):
        print("Loading ESMFold...")
        self.tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
        self.m = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1",
                                                      low_cpu_mem_usage=True).to(device).eval()
    def _f(self, s):
        enc = self.tok([s], return_tensors="pt", add_special_tokens=False).to(device)
        with torch.no_grad(): o = self.m(**enc)
        raw = float(np.mean(o.plddt.cpu().numpy()))
        return (raw*100.0 if raw <= 1.5 else raw), (float(o.ptm.item()) if hasattr(o,"ptm") else 0.0)
    def fold_one(self, seq, max_len=300):
        c = "".join(a for a in seq if a in VALID_AA)
        if len(c) < 10: return 0.0, 0.0, False
        w = c[:max_len]
        try:
            p, t = self._f(w); return p, t, True
        except RuntimeError: clear_gpu()
        half = max(10, len(w)//2)
        if half < len(w):
            try:
                p, t = self._f(w[:half]); return p, t, True
            except RuntimeError: clear_gpu()
        return 0.0, 0.0, False

def fold_records(recs, ev):
    for r in recs:
        p, t, ok = ev.fold_one(r["sequence"])
        r["plddt"], r["ptm"], r["fold_ok"] = p, t, ok
        r["collapse"] = int(0.0 < p < 60.0)
        r["repetition_score"] = repetition_score(r["sequence"])
        r["utility_score"] = utility_score(p, t) if p > 0 else 0.0
    return recs

print("Scoring + evaluator ready.")


Scoring + evaluator ready.


In [4]:
# --- RITA loader and manual sampling loop, verbatim from 32/33/44. All five known issues. ---

def load_rita(device):
    import transformers.modeling_utils as _mu
    _hm = "mark_tied_weights_as_initialized" in _mu.PreTrainedModel.__dict__
    _om = _mu.PreTrainedModel.__dict__.get("mark_tied_weights_as_initialized")
    _ha = "all_tied_weights_keys" in _mu.PreTrainedModel.__dict__
    _oa = _mu.PreTrainedModel.__dict__.get("all_tied_weights_keys")
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = lambda self: None
    _mu.PreTrainedModel.all_tied_weights_keys = property(lambda self: {})
    try:
        tok = AutoTokenizer.from_pretrained("lightonai/RITA_s")
        m = AutoModelForCausalLM.from_pretrained("lightonai/RITA_s", trust_remote_code=True,
                                                 torch_dtype=torch.float32).to(device).eval()
    finally:
        if _hm: _mu.PreTrainedModel.mark_tied_weights_as_initialized = _om
        else: del _mu.PreTrainedModel.mark_tied_weights_as_initialized
        if _ha: _mu.PreTrainedModel.all_tied_weights_keys = _oa
        else: del _mu.PreTrainedModel.all_tied_weights_keys
    if tok.pad_token_id is None and tok.eos_token_id is not None:
        tok.pad_token = tok.eos_token
    return tok, m

def manual_generate(model, input_ids, max_len, temperature=1.2, steer=None, layer_module=None):
    # Steering hook is registered by the CALLER around this loop, so it applies to every
    # forward pass inside it automatically.
    gen = input_ids
    with torch.no_grad():
        for _ in range(max_len - input_ids.shape[1]):
            logits = model(gen).logits[:, -1, :] / temperature
            nxt = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            gen = torch.cat([gen, nxt], dim=1)
    return gen

print(f"Loading RITA-small on {device}...")
rita_tok, rita = load_rita(device)
N_LAYERS = len(rita.transformer.layers)
print(f"RITA has {N_LAYERS} layers; using layer {RITA_LAYER} ({RITA_LAYER/(N_LAYERS-1):.0%} depth).")

cand_prefixes = build_prefix_pool(N_CANDIDATES, seed=606)
torch.manual_seed(606)
candidates = []
print(f"=== Generating N={N_CANDIDATES} natural candidates (manual loop — slower) ===")
for p in cand_prefixes:
    ids = rita_tok(p, return_tensors="pt").input_ids.to(device)
    out = manual_generate(rita, ids, max_len=50)
    s = rita_tok.decode(out[0], skip_special_tokens=True).replace(" ", "")
    candidates.append({"prompt": p, "sequence": s,
                       "gen_only": s[len(p):] if s.startswith(p) else s})
clear_gpu()

print("=== Freeing RITA while ESMFold folds the pool ===")
del rita; clear_gpu()

ev = SafeEvaluator()
print("Folding candidate pool...")
candidates = fold_records(candidates, ev)
del ev; clear_gpu()

valid = [r for r in candidates if r["fold_ok"]]
pool_collapse = float(np.mean([r["collapse"] for r in valid]))
pool_plddt = float(np.mean([r["plddt"] for r in valid]))
print(f"\n{len(valid)}/{len(candidates)} folded. Natural collapse {pool_collapse:.1%}, "
      f"mean pLDDT {pool_plddt:.2f}")
print(f"(§1k recorded 84.0% pool collapse / 52.29 pLDDT; §3f 89.5%. Close values confirm")
print(f" comparability. NOTE the ~90% ceiling — this is why pLDDT, not collapse rate, is the")
print(f" primary outcome in this notebook.)")


Loading RITA-small on cuda...


config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

The repository lightonai/RITA_s contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/lightonai/RITA_s .
 You can inspect the repository content at https://hf.co/lightonai/RITA_s.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


rita_configuration.py:   0%|          | 0.00/861 [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/lightonai/RITA_s:
- rita_configuration.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

pytorch_model.bin:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/8.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding candidate pool...

200/200 folded. Natural collapse 84.5%, mean pLDDT 52.13
(§1k recorded 84.0% pool collapse / 52.29 pLDDT; §3f 89.5%. Close values confirm
 comparability. NOTE the ~90% ceiling — this is why pLDDT, not collapse rate, is the
 primary outcome in this notebook.)


In [5]:
# --- Utility-matched vector at layer 3, scaled to MATCHED alpha_rel (not to 583.998). ---

QUANTILE, UTIL_TOL = 0.30, 0.05
srt = sorted(valid, key=lambda r: r["repetition_score"])
n_side = max(10, int(len(srt) * QUANTILE))
d_minus_raw, d_plus_raw = srt[:n_side], srt[-n_side:]

def utility_match(a, b, tol, iters=200):
    a, b = list(a), list(b)
    for _ in range(iters):
        ma, mb = np.mean([r["utility_score"] for r in a]), np.mean([r["utility_score"] for r in b])
        gap = ma - mb
        if abs(gap) <= tol or min(len(a), len(b)) <= 15: break
        if gap > 0: a.sort(key=lambda r: -r["utility_score"]); a.pop(0)
        else: b.sort(key=lambda r: r["utility_score"]); b.pop(0)
    return a, b

d_plus, d_minus = utility_match(d_plus_raw, d_minus_raw, UTIL_TOL)
print(f"Utility-matched: D+ n={len(d_plus)}, D- n={len(d_minus)}, gap "
      f"{abs(np.mean([r['utility_score'] for r in d_plus]) - np.mean([r['utility_score'] for r in d_minus])):.3f}")

print(f"\nReloading RITA to extract activations and measure ‖h‖...")
rita_tok, rita = load_rita(device)
layer_mod = rita.transformer.layers[RITA_LAYER]

def mean_activation_hooked(seqs):
    cap, acts = {}, []
    def hook(m, i, o):
        cap["h"] = (o[0] if isinstance(o, (tuple, list)) else o).detach()
    handle = layer_mod.register_forward_hook(hook)
    try:
        for s in seqs:
            ids = rita_tok(s, return_tensors="pt", truncation=True, max_length=256).input_ids.to(device)
            cap.clear()
            with torch.no_grad(): rita(ids)
            if "h" in cap:
                acts.append(cap["h"].float().mean(dim=1).squeeze(0).cpu())
    finally:
        handle.remove()
    return torch.stack(acts)

pos = mean_activation_hooked([r["sequence"] for r in d_plus])
neg = mean_activation_hooked([r["sequence"] for r in d_minus])
v_raw = pos.mean(dim=0) - neg.mean(dim=0)
raw_norm = v_raw.norm().item()

h_rita = measure_resid_norm_hooked(rita, rita_tok, probe_seqs, layer_mod)
matched_norm = anchor_alpha_rel * h_rita
v_unit = (v_raw * (matched_norm / raw_norm)).to(device)

print(f"\nLayer {RITA_LAYER}: raw v_L norm = {raw_norm:.4f}")
print(f"  RITA ‖h‖ = {h_rita:.2f}   (34 measured 23.69 — close value confirms stability)")
print(f"  matched norm for alpha_rel={anchor_alpha_rel:.4f}  ->  {matched_norm:.4f}")
print(f"  §1k used {REFERENCE_NORM:.1f} here = {REFERENCE_NORM/matched_norm:.0f}x HARDER than matched")
print(f"  sanity: matched_norm/‖h‖ = {matched_norm/h_rita:.4f} (must equal anchor)")


Utility-matched: D+ n=60, D- n=60, gap 0.020

Reloading RITA to extract activations and measure ‖h‖...


The repository lightonai/RITA_s contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/lightonai/RITA_s .
 You can inspect the repository content at https://hf.co/lightonai/RITA_s.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]


Layer 3: raw v_L norm = 3.1067
  RITA ‖h‖ = 23.88   (34 measured 23.69 — close value confirms stability)
  matched norm for alpha_rel=0.2000  ->  4.7756
  §1k used 584.0 here = 122x HARDER than matched
  sanity: matched_norm/‖h‖ = 0.2000 (must equal anchor)


In [6]:
# --- Steering across the wide dose ladder. Hook registered around the manual loop. ---

def generate_steered(prompts, vec, seed):
    torch.manual_seed(seed)
    v = None if vec is None else vec.to(device)
    def hook(m, i, o):
        if v is None: return o
        if isinstance(o, tuple): return (o[0] + v,) + o[1:]
        return o + v
    recs = []
    for p in prompts:
        handle = layer_mod.register_forward_hook(hook)
        ids = rita_tok(p, return_tensors="pt").input_ids.to(device)
        out = manual_generate(rita, ids, max_len=50)
        handle.remove()
        s = rita_tok.decode(out[0], skip_special_tokens=True).replace(" ", "")
        g = s[len(p):] if s.startswith(p) else s
        recs.append({"prompt": p, "sequence": s, "gen_only": g, "entropy": calculate_entropy(g)})
    clear_gpu()
    return recs

steer_prefixes = build_prefix_pool(len(DOSES) * N_PER_CONDITION, seed=607)
conditions = {}
for i, d in enumerate(DOSES):
    name = f"L{RITA_LAYER}_MATCHED_{d:g}x"
    lo, hi = i * N_PER_CONDITION, (i + 1) * N_PER_CONDITION
    print(f"=== {name} (alpha_rel {d*anchor_alpha_rel:.4f}) ===")
    conditions[name] = generate_steered(steer_prefixes[lo:hi],
                                        None if d == 0 else v_unit * d, seed=1200 + i)

print("\n=== Freeing RITA from GPU ===")
del rita; clear_gpu()

ev2 = SafeEvaluator()
for n in conditions:
    print(f"Folding {n}...")
    conditions[n] = fold_records(conditions[n], ev2)
del ev2; clear_gpu()


=== L3_MATCHED_0x (alpha_rel 0.0000) ===
=== L3_MATCHED_1x (alpha_rel 0.2000) ===
=== L3_MATCHED_2x (alpha_rel 0.4000) ===
=== L3_MATCHED_4x (alpha_rel 0.8000) ===
=== L3_MATCHED_8x (alpha_rel 1.6000) ===

=== Freeing RITA from GPU ===
Loading ESMFold...


Loading weights:   0%|          | 0/4533 [00:00<?, ?it/s]

EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status     | 
-----------------------------------+------------+-
esm.embeddings.position_ids        | UNEXPECTED | 
esm.contact_head.regression.weight | MISSING    | 
esm.contact_head.regression.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Folding L3_MATCHED_0x...
Folding L3_MATCHED_1x...
Folding L3_MATCHED_2x...
Folding L3_MATCHED_4x...
Folding L3_MATCHED_8x...


In [7]:
# --- Results. pLDDT is PRIMARY (continuous, unsaturated); collapse is SECONDARY (~90% ceiling). ---

def wilson(k, n, z=1.959963985):
    if n == 0: return (0.0, 0.0)
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return (max(0.0, c-h), min(1.0, c+h))

ctrl_name = f"L{RITA_LAYER}_MATCHED_0x"
ctrl_plddt = [r["plddt"] for r in conditions[ctrl_name] if r["fold_ok"]]
ctrl_k = sum(r["collapse"] for r in conditions[ctrl_name])
ctrl_n = len(conditions[ctrl_name])

print("=" * 100)
print(f"RITA AT MATCHED alpha_rel (anchor {anchor_alpha_rel:.4f}) — layer {RITA_LAYER}")
print("=" * 100)
print(f"{'dose':>7s} {'alpha_rel':>10s} {'FoldOK':>7s} {'mean pLDDT':>11s} {'MWU p':>9s} "
      f"{'entropy':>8s} {'collapse':>9s} {'Fisher p':>9s}")
print("-" * 88)

rows = []
for d, name in zip(DOSES, conditions):
    recs = conditions[name]
    n = len(recs)
    ok = sum(1 for r in recs if r["fold_ok"])
    pl = [r["plddt"] for r in recs if r["fold_ok"]]
    k = sum(r["collapse"] for r in recs)
    ent = float(np.mean([r["entropy"] for r in recs]))
    if d == 0.0:
        p_mwu = p_fis = float("nan")
    else:
        p_mwu = float(mannwhitneyu(pl, ctrl_plddt, alternative="less").pvalue) if pl else float("nan")
        _, p_fis = fisher_exact([[k, n-k], [ctrl_k, ctrl_n-ctrl_k]])
    lo, hi = wilson(k, n)
    rows.append({"dose": d, "alpha_rel": d*anchor_alpha_rel, "n": n, "fold_ok": ok,
                 "mean_plddt": float(np.mean(pl)) if pl else float("nan"),
                 "mwu_p": p_mwu, "entropy": ent, "collapsed": k, "collapse_rate": k/n,
                 "ci_lo": lo, "ci_hi": hi, "fisher_p": p_fis})
    print(f"{d:6.1f}x {d*anchor_alpha_rel:10.4f} {ok:3d}/{n:<3d} "
          f"{(np.mean(pl) if pl else float('nan')):11.2f} "
          f"{p_mwu:9.4f} {ent:8.3f} {k:4d}/{n:<4d} {p_fis:9.4f}")

df = pd.DataFrame(rows)
print()
print("MWU p = Mann-Whitney U, one-sided (is pLDDT LOWER than control?). Primary test.")
print("Fisher p = collapse rate vs control. Secondary — saturates near RITA's ~90% ceiling.")

print()
print("Read against §1k's ORIGINAL absolute-norm run (alpha_rel 24.65 — 123x harder):")
print("  CONTROL 90.0% -> L3_1x 96.0% -> L3_2x 100.0%  (collapse rate)")
print(f"This run: CONTROL {rows[0]['collapse_rate']:.1%} -> " +
      " -> ".join(f"{r['dose']:g}x {r['collapse_rate']:.1%}" for r in rows[1:]))


RITA AT MATCHED alpha_rel (anchor 0.2000) — layer 3
   dose  alpha_rel  FoldOK  mean pLDDT     MWU p  entropy  collapse  Fisher p
----------------------------------------------------------------------------------------
   0.0x     0.0000  50/50        52.64       nan    4.022   39/50         nan
   1.0x     0.2000  50/50        53.13    0.6816    3.987   41/50      0.8031
   2.0x     0.4000  50/50        51.09    0.1497    4.084   44/50      0.2869
   4.0x     0.8000  50/50        52.77    0.4007    4.055   41/50      0.8031
   8.0x     1.6000  50/50        52.57    0.4357    4.069   40/50      1.0000

MWU p = Mann-Whitney U, one-sided (is pLDDT LOWER than control?). Primary test.
Fisher p = collapse rate vs control. Secondary — saturates near RITA's ~90% ceiling.

Read against §1k's ORIGINAL absolute-norm run (alpha_rel 24.65 — 123x harder):
  CONTROL 90.0% -> L3_1x 96.0% -> L3_2x 100.0%  (collapse rate)
This run: CONTROL 78.0% -> 1x 82.0% -> 2x 88.0% -> 4x 82.0% -> 8x 80.0%


In [8]:
# --- Verdict + persistence. ---
sig_plddt = [r for r in rows[1:] if not np.isnan(r["mwu_p"]) and r["mwu_p"] < 0.05]
sig_coll = [r for r in rows[1:] if not np.isnan(r["fisher_p"]) and r["fisher_p"] < 0.05]
ent_ctrl = rows[0]["entropy"]
ent_move = [r for r in rows[1:] if abs(r["entropy"] - ent_ctrl) > 0.15]

print("=" * 92)
print("VERDICT")
print("=" * 92)
if sig_plddt:
    first = min(sig_plddt, key=lambda r: r["dose"])
    print(f"RITA DOES degrade at matched relative push — first significant by pLDDT at "
          f"{first['dose']:g}x (alpha_rel {first['alpha_rel']:.3f}, p={first['mwu_p']:.4f}).")
    print(f"  Compare thresholds: ProtGPT2 alpha_50 = 0.213 (§1n); ZymCTRL first significant at")
    print(f"  3x = alpha_rel 0.60 (§1h-THRESHOLD); RITA here at alpha_rel {first['alpha_rel']:.3f}.")
    print(f"  -> Report RITA's own threshold multiple explicitly. Three models with three")
    print(f"     measured thresholds is a real cross-model result, replacing the retracted")
    print(f"     '5 models show the same pattern' claim with a quantified one.")
else:
    print(f"RITA shows NO significant pLDDT degradation at any dose up to {max(DOSES):g}x matched")
    print(f"(alpha_rel {max(DOSES)*anchor_alpha_rel:.2f}) — well past ZymCTRL's 3x threshold.")
    print(f"  Two readings, and this run cannot separate them: (a) RITA is genuinely robust to")
    print(f"  relative perturbation; (b) RITA's ~90% natural collapse leaves too little headroom")
    print(f"  for any effect to register even on the continuous pLDDT readout. Check the mean")
    print(f"  pLDDT column — if it is already near the collapse threshold at 0x, (b) is likely.")

print()
if sig_coll and not sig_plddt:
    print("NOTE: collapse rate moved but pLDDT did not — treat with suspicion at a ~90% ceiling.")
elif sig_plddt and not sig_coll:
    print("NOTE: pLDDT moved but collapse rate did not. Expected — this is exactly why pLDDT is")
    print("      the primary outcome here; the binary metric is saturated on RITA.")

print()
moved_labels = ", ".join("{:g}x".format(r["dose"]) for r in ent_move) if ent_move else "NONE"
print(f"Entropy: control {ent_ctrl:.3f}; doses moving >0.15 from control: {moved_labels}")
if not ent_move:
    print("  ==> Steering never achieved measurable repetition reduction at ANY matched dose.")
    print("      Same pattern as ZymCTRL/p-IgGen: at doses comparable to ProtGPT2's, steering on")
    print("      these models does neither good nor harm. Worth stating plainly — the method's")
    print("      intended EFFECT is as absent as its damage.")
else:
    print("  ==> Entropy moved at some doses — check whether it moves BEFORE pLDDT falls. A dose")
    print("      where entropy rises while pLDDT holds would be the first evidence anywhere in")
    print("      this project of a genuine safe operating window.")

seq_rows = []
for d, name in zip(DOSES, conditions):
    for i, r in enumerate(conditions[name]):
        seq_rows.append({"condition": name, "dose": d, "alpha_rel": d*anchor_alpha_rel, "idx": i,
                         "prompt": r["prompt"], "sequence": r["sequence"],
                         "gen_length": len(r["gen_only"]), "entropy": r["entropy"],
                         "plddt": r["plddt"], "ptm": r["ptm"], "fold_ok": r["fold_ok"],
                         "collapse": r["collapse"]})
pd.DataFrame(seq_rows).to_csv("rita_matched_alpha_rel_sequences.csv", index=False)
df.to_csv("rita_matched_alpha_rel_summary.csv", index=False)
pd.DataFrame([{"anchor_alpha_rel": anchor_alpha_rel, "h_protgpt2_l12": h_protgpt2,
               "h_rita": h_rita, "matched_norm": matched_norm, "raw_vL_norm": raw_norm,
               "layer": RITA_LAYER, "reference_norm_used_by_1k": REFERENCE_NORM,
               "pool_collapse": pool_collapse, "pool_plddt": pool_plddt}]
             ).to_csv("rita_matched_alpha_rel_calibration.csv", index=False)
print("\nSaved: rita_matched_alpha_rel_{sequences,summary,calibration}.csv")
print("Update locked-results.md §1k with this run — RITA was the last unrepaired model.")


VERDICT
RITA shows NO significant pLDDT degradation at any dose up to 8x matched
(alpha_rel 1.60) — well past ZymCTRL's 3x threshold.
  Two readings, and this run cannot separate them: (a) RITA is genuinely robust to
  relative perturbation; (b) RITA's ~90% natural collapse leaves too little headroom
  for any effect to register even on the continuous pLDDT readout. Check the mean
  pLDDT column — if it is already near the collapse threshold at 0x, (b) is likely.


Entropy: control 4.022; doses moving >0.15 from control: NONE
  ==> Steering never achieved measurable repetition reduction at ANY matched dose.
      Same pattern as ZymCTRL/p-IgGen: at doses comparable to ProtGPT2's, steering on
      these models does neither good nor harm. Worth stating plainly — the method's
      intended EFFECT is as absent as its damage.

Saved: rita_matched_alpha_rel_{sequences,summary,calibration}.csv
Update locked-results.md §1k with this run — RITA was the last unrepaired model.
